# Steric blocking demo

**What it is.** A PRC1 head that is bound to a microtubule can make it harder for another PRC1 to bind close to it. In this model, an occupied site lowers the binding rate of the free sites next to it on the same microtubule. The effect gets weaker with distance, so closer sites are less likely to be bound than farther ones.

**Two versions.**
- **Graded:** each occupied neighbor multiplies a free site's binding weight by `1 - (1 - c) * exp(-d / decay)`. Nearby sites are reduced a lot, the effect is mostly gone at the range, and it drops to zero just beyond it.
- **Hard:** sites within the range of an occupied site cannot be bound at all.

This is a working hypothesis. No source in the project papers gives a measured distance law for PRC1, so the range and the decay length are free parameters.

## The range parameter

`steric_range_nm` is set in nanometers. It is converted to a whole number of lattice sites in one place, `range_nm_to_sites`.

**The default is 0 nm. That turns the feature off** and reproduces the earlier model exactly. A range smaller than one site gives a warning and turns the feature off, so a small range is never silently rounded to a one-site effect.

In [ ]:
import io
import json
import sys
import tempfile
import time
import warnings
from pathlib import Path

import numpy as np
from IPython.display import Image, display


def find_repo_root():
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "python_version" / "steric_sweep.py").exists():
            return folder
    raise FileNotFoundError("Run this notebook from inside the repository.")


def show(fig):
    # render a matplotlib figure as a PNG in the notebook (works with any plotting backend)
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=110)
    display(Image(buffer.getvalue()))


REPO = find_repo_root()
sys.path.insert(0, str(REPO / "python_version"))

import steric_sweep as ss
from prc1_state import range_nm_to_sites

In [ ]:
spacing = 0.2   # nm per site on the current lattice
for nm in (0, 0.1, 2, 8, 32):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        sites = range_nm_to_sites(nm, spacing)
    note = "   (warns: below one site, feature off)" if caught else ""
    print(f"{nm:>5} nm -> {sites:>3} sites{note}")

The lattice here has 0.2 nm between sites, so one tubulin dimer (about 8 nm) is 40 sites. The range is given in nm so that it means the same thing if the lattice spacing changes later.

## One occupied site

This is the model's binding weight for a free site at distance d from a single occupied site (1 means unaffected). It is computed from the model itself, with no simulation.

In [ ]:
cfg = ss.SweepConfig()
profiles = {}
for range_nm, mode in [(0.0, "off")] + [(r, m) for r in (2.0, 8.0, 16.0) for m in ("graded", "hard")]:
    distance_nm, weight, _ = ss.binding_profile(cfg, range_nm, mode)
    profiles[(range_nm, mode)] = dict(profile_distance_nm=distance_nm, profile_weight=weight)
show(ss.plot_binding_profiles(profiles, max_nm=30))

## A small live run

This runs a short microtubule (300 nm) with 20 seeds and three settings: steric off, and an 8 nm range in both versions. It takes well under a minute. The density is set higher than in the tests so a short run has enough heads to compare.

In [ ]:
demo = ss.SweepConfig(ranges_nm=(0.0, 8.0), modes=("graded", "hard"), seeds=tuple(range(20)),
                      end_time=10.0, max_steps=5000, max_distance_nm=40.0)
demo.lattice = dict(demo.lattice, microtubule_length=300.0, microtubule_offset=100)
demo.rates = dict(demo.rates, initial_binding_rate_per_site=2e-3)

start = time.time()
with tempfile.TemporaryDirectory() as tmp:
    ss.run_sweep(demo, tmp, verbose=False)
    demo_results = ss.load_results(tmp)
print(f"finished in {time.time() - start:.0f} s")
ss.print_summary(demo_results)

In [ ]:
show(ss.plot_attach_profiles(demo_results, max_nm=30))

For each new first head, the plot compares how close it landed to an already bound head with what uniform binding would give at that moment. A value of 1 means no steric effect and 0 means no new heads land that close. Lines are the mean over 20 seeds and bands are one standard error. With steric off the line sits at 1. With steric on, fewer new heads land close to a bound head, and with the hard version none land inside the range.

## The saved sweep

The full sweep used the production lattice (5000 nm, 0.2 nm sites) with 30 seeds per setting. It was run with `python python_version/steric_sweep.py run` and saved to `results/steric_sweep/`, which is not committed. The plots below load those files and do not run any simulation. To rebuild the figures, run `python python_version/steric_sweep.py plot`.

Lines are the mean over seeds and bands are one standard error. The gray line is steric off. Light to dark blue means a longer range. Solid lines are the graded version and dashed lines are the hard version.

In [ ]:
SWEEP_DIR = REPO / "results" / "steric_sweep"
results = None
if (SWEEP_DIR / "manifest.json").exists():
    results = ss.load_results(SWEEP_DIR)
    manifest = json.loads((SWEEP_DIR / "manifest.json").read_text())
    n_seeds = len(next(iter(results.values()))["seed"])
    print(f"Loaded {len(results)} settings with {n_seeds} seeds each (commit {str(manifest['git_commit'])[:7]}).")
    ss.print_summary(results)
else:
    print("No saved sweep found. Create one with: python python_version/steric_sweep.py run")

In [ ]:
if results:
    show(ss.plot_attach_profiles(results))

In [ ]:
if results:
    show(ss.plot_pair_profiles(results))

In [ ]:
if results:
    show(ss.plot_summary(results))

## What the sweep showed

For the 30 seed sweep behind this report:

- **New first heads avoid bound heads.** The comparison is with uniform binding at the same occupancy, so it does not depend on how crowded the microtubule is. With steric off it gives 1.0, as it should. With a graded 8 nm range, new first heads land within 2 nm of a bound head only 0.35 as often as uniform binding would give, and within 8 nm 0.73 as often. With hard exclusion none land inside the range.
- **Occupancy near a head drops, but not to zero.** The same-lane pair density (1 means no correlation) in the 0 to 4 nm bin is 0.89 with steric off, 0.72 for a graded 8 nm range and 0.41 for hard 8 nm. It is not zero for the hard version because second heads ignore the penalty (see the limitations).
- **Nothing changes beyond the range.** At 40 to 64 nm the pair density is about 1.0 in every setting.
- **The pair density is measured against even placement inside the overlap.** Nearly all heads sit in the 3000 nm overlap of the two microtubules, which is 60% of each lane. Measured against the whole lane, the pair density would read about 1.7 even with steric off. It is confinement, not clustering. The small deficit at 0 to 4 nm with steric off (0.89) is not caused by steric blocking and its cause is not established.
- **Fewer PRC1 end up bound.** After 30 time units the mean count is 180 with steric off, 161 for a graded 8 nm range and 124 for hard 8 nm. A graded 2 nm range is not distinguishable from off.

## Limitations

- **First head only.** The penalty applies to the first head of a PRC1. A second head can still land inside another head's range on the other lane.
- **No cooperativity or hopping.** Steric blocking cannot be combined with either yet, and the code raises an error if you try.
- **The lattice.** Sites are 0.2 nm apart here, while the CyLaKS papers use 8 nm. Fitted per-site rates only make sense for this lattice, and one dimer is 40 sites. This is unresolved and logged in `ISSUES.md`.
- **The distance law is a hypothesis.** The range, decay length and contact factor have no verified source. Read the results as sensitivity to these parameters, not as predictions.
- **Not a steady state.** Runs stop at 30 time units, when the PRC1 count with steric off is still rising (a longer run reached 1008 at 240 time units). The binding rate was chosen so there are enough heads to measure and is not fitted to data. Any rate called fitted would only be valid on this 0.2 nm lattice.